# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **what are the population trends in Pittsburgh over the past 50 years?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-16 11:13:28 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-16T11:13:28.626730")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `Pittsburgh population trends historical census`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 5 datasets matching 'Pittsburgh population trends historical census'

1. **Pittsburgh American Community Survey Data 2015 - Household Types**
   ID: `pittsburgh-american-community-survey-data-household-types`
   The data on relationship to householder were derived from answers to Question 2 in the 2015 American Community Survey (ACS), which was asked of all people in housing units. The que
   - Household Type (American Indian and Alaska Native Alone) (CSV) [DataStore] ID: `5fb99124-f92f-491e-821e-06fa7eeb74e9`
   - Household Type, Asian Alone (CSV) [DataStore] ID: `70610c6d-822e-47fb-bd2
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'Pittsburgh population trends historical census', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `population census demographics Pittsburgh`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 10 datasets matching 'population census demographics Pittsburgh'

1. **Pittsburgh American Community Survey 2015 - Miscellaneous Data**
   ID: `pittsburgh-american-community-survey-2015-miscellaneous-data`
   Miscellaneous Reports from the 2015 American Community Survey Report  ___Support for Health Equity datasets and tools provided by [Amazon Web Services (AWS)](https://aws.amazon.com
   - Aggregate Household Income in the Past 12 Months (CSV) [DataStore] ID: `34842307-0da6-458a-9df3-a09ab3e3a489`
   - Types of Health Insurance Coverage, By Age (CSV) [DataStore] ID: `c9fd5859-ed34-4151
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'population census demographics Pittsburgh', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `b7156251-6036-4b68-ad2a-95566c84343e`
**Limit:** 5

**Result preview:**
```
Resource: b7156251-6036-4b68-ad2a-95566c84343e
Total records: 90
Loaded: 5
Fields (19): Neighborhood, Sector #, Population (2010), Miles of Major Roads, Total Street Miles, Street Density (st. mi/area sq. mi), # Sets of Steps, # Step Treads, Res. Permit Parking Area(s), Total Working Pop. (Age 16+) (2010), Commute to Work: Drive Alone (2010), Commute to Work: Carpool/Vanpool (2010), Commute to Work: Public Transportation (2010), Commute to Work: Taxi (2010), Commute to Work: Motorcycle (2010), Commute to Work: Bicycle (2010), Commute to Work: Walk (2010), Commute to Work: Other (2010), Work at
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": 'b7156251-6036-4b68-ad2a-95566c84343e', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 4: Search for Datasets

**Search query:** `Pittsburgh neighborhood population change historical`

**Result preview:**
```
Found 13 datasets matching 'Pittsburgh neighborhood population change historical'

1. **Pittsburgh Neighborhood Atlas, 1977**
   ID: `pittsburgh-neighborhood-atlas-1977`
   This compilation includes five historical datasets that are part of the University of Pittsburgh Library collection. The datasets were transcribed from The Pittsburgh Neighborhood 
   - Original Documents at Historic Pittsburgh (HTML) ID: `5620acde-55b8-46bd-8c97-23393627ef7c`
   - Crime Rate, 1973-1975 (CSV) [DataStore] ID: `967cad5c-ea8f-4558-9dc9-8d3f7a3b046c`
   - Data Dictionary for Crime Rate, 1973-1975 (PDF) ID: `a51
```


In [ ]:
# Step 4: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'Pittsburgh neighborhood population change historical', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 5: Load Data from Resource

**Resource ID:** `a8414ed5-c50f-417e-bb67-82b734660da6`
**Limit:** 5

**Result preview:**
```
Resource: a8414ed5-c50f-417e-bb67-82b734660da6
Total records: 89
Loaded: 5
Fields (31): Neighborhood, 2010_Total_Population, 2010_OneRace_Population, 2010_White_Alone_Population, 2010_Black_Alone_Population, 2010_American_Indian_and_Alaska_Native_alone_population, 2010_Asian_Alone_Population, 2010_Native_Hawaiian_and_Other_Pacific_Islander_alone_Populatio, 2010_Any_other_single_race_alone_Popualtion, 2010_Population_two_or_more_races, 2010_Hispanic_or_Latino_Population, 2020_Total_Population, 2020_OneRace_Population, 2020_White_Alone_Population, 2020_Black_Alone_Population, 2020_American_India
```


In [ ]:
# Step 5: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": 'a8414ed5-c50f-417e-bb67-82b734660da6', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 6: Dataset Details

**Dataset:** `1984-community-profiles-a-descriptive-picture-of-communities-in-allegheny-county`

**Result preview:**
```
# 1984 Community Profiles: A Descriptive Picture of Communities in Allegheny County

This compilation includes 8 historical datasets, part of the collection of the Pittsburgh University Library System. Prepared by the Health and Welfare Planning Association, and funded by the Richard King Melon Foundation, the profiles were intended to inform the people, organizations and businesses of Allegheny County about the characteristics of the communities in which they lived, worked and played so they could make better decisions.

___Support for Health Equity datasets and tools provided by [Amazon We
```


In [ ]:
# Step 6: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": '1984-community-profiles-a-descriptive-picture-of-communities-in-allegheny-county'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 7: Dataset Details

**Dataset:** `pittsburgh-neighborhood-atlas-1977`

**Result preview:**
```
# Pittsburgh Neighborhood Atlas, 1977

This compilation includes five historical datasets that are part of the University of Pittsburgh Library collection. The datasets were transcribed from The Pittsburgh Neighborhood Atlas, published in 1977. The atlas was prepared by the Pittsburgh Neighborhood Alliance.  The information provides an insight into the neighborhoods conditions and the direction in which they were moving at the time of preparation. Much of the material describing neighborhood characteristics came from figures compiled for smaller areas: voting districts or census blocks.
The f
```


In [ ]:
# Step 7: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'pittsburgh-neighborhood-atlas-1977'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 8: Load Data from Resource

**Resource ID:** `8e2b239e-4a46-49f9-bd13-2566f3c64d28`
**Limit:** 10

**Result preview:**
```
Resource: 8e2b239e-4a46-49f9-bd13-2566f3c64d28
Total records: 218
Loaded: 10
Fields (20): Community, pop_total, %race_white, %race_black, %race_other, %age_0-4, %age_5-18, %age_19-44, %age_45-64, %age_65+, %fem_married, %fem_single, %fem_sep, %fem_widowed, %fem_under15_married, %male_married, %male_single, %male_sep, %male_widowed, %male_under15_married

Sample (10 rows):

        Community pop_total %race_white %race_black %race_other %age_0-4 %age_5-18 %age_19-44 %age_45-64 %age_65+ %fem_married %fem_single %fem_sep %fem_widowed %fem_under15_married %male_married %male_single %male_sep %male
```


In [ ]:
# Step 8: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '8e2b239e-4a46-49f9-bd13-2566f3c64d28', "limit": 10}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 9: SQL Analysis Query

**SQL:**
```sql
SELECT SUM("2010_Total_Population"::int) as total_2010, SUM("2020_Total_Population"::int) as total_2020, SUM("Change_2010_to_2020_Total_Population"::int) as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6"
```

**Result preview:**
```
SQL: SELECT SUM("2010_Total_Population"::int) as total_2010, SUM("2020_Total_Population"::int) as total_2020, SUM("Change_2010_to_2020_Total_Population"::int) as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6"
Rows: 1
Columns: total_2010, total_2020, change

 total_2010  total_2020  change
     305704      302971   -2733
```


In [ ]:
# Step 9: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT SUM("2010_Total_Population"::int) as total_2010, SUM("2020_Total_Population"::int) as total_2020, SUM("Change_2010_to_2020_Total_Population"::int) as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6"'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 10: Load Data from Resource

**Resource ID:** `8e2b239e-4a46-49f9-bd13-2566f3c64d28`
**Limit:** 5
**Filters:** `{"Community": "Pittsburgh"}`

**Result preview:**
```
Resource: 8e2b239e-4a46-49f9-bd13-2566f3c64d28
Total records: 1
Loaded: 1
Fields (20): Community, pop_total, %race_white, %race_black, %race_other, %age_0-4, %age_5-18, %age_19-44, %age_45-64, %age_65+, %fem_married, %fem_single, %fem_sep, %fem_widowed, %fem_under15_married, %male_married, %male_single, %male_sep, %male_widowed, %male_under15_married

Sample (1 rows):

 Community pop_total %race_white %race_black %race_other %age_0-4 %age_5-18 %age_19-44 %age_45-64 %age_65+ %fem_married %fem_single %fem_sep %fem_widowed %fem_under15_married %male_married %male_single %male_sep %male_widowed %m
```


In [ ]:
# Step 10: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '8e2b239e-4a46-49f9-bd13-2566f3c64d28', "limit": 5, "filters": {"Community": "Pittsburgh"}}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 11: SQL Analysis Query

**SQL:**
```sql
SELECT "Neighborhood", "2010_Total_Population"::int as pop_2010, "2020_Total_Population"::int as pop_2020, "Change_2010_to_2020_Total_Population"::int as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6" ORDER BY "Change_2010_to_2020_Total_Population"::int DESC LIMIT 10
```

**Result preview:**
```
SQL: SELECT "Neighborhood", "2010_Total_Population"::int as pop_2010, "2020_Total_Population"::int as pop_2020, "Change_2010_to_2020_Total_Population"::int as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6" ORDER BY "Change_2010_to_2020_Total_Population"::int DESC LIMIT 10
Rows: 10
Columns: Neighborhood, pop_2010, pop_2020, change

                        Neighborhood  pop_2010  pop_2020  change
Central Business District (Downtown)      3629      5477    1848
                           Shadyside     13915     15317    1402
                      Strip District       616      1845    1229
   
```


In [ ]:
# Step 11: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "Neighborhood", "2010_Total_Population"::int as pop_2010, "2020_Total_Population"::int as pop_2020, "Change_2010_to_2020_Total_Population"::int as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6" ORDER BY "Change_2010_to_2020_Total_Population"::int DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 12: SQL Analysis Query

**SQL:**
```sql
SELECT "Neighborhood", "2010_Total_Population"::int as pop_2010, "2020_Total_Population"::int as pop_2020, "Change_2010_to_2020_Total_Population"::int as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6" ORDER BY "Change_2010_to_2020_Total_Population"::int ASC LIMIT 10
```

**Result preview:**
```
SQL: SELECT "Neighborhood", "2010_Total_Population"::int as pop_2010, "2020_Total_Population"::int as pop_2020, "Change_2010_to_2020_Total_Population"::int as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6" ORDER BY "Change_2010_to_2020_Total_Population"::int ASC LIMIT 10
Rows: 10
Columns: Neighborhood, pop_2010, pop_2020, change

            Neighborhood  pop_2010  pop_2020  change
      Marshall-Shadeland      6043      4046   -1997
Lincoln-Lemington-Belmar      4883      3636   -1247
         Terrace Village      3228      2320    -908
                   Bluff      6600      5882    -718
```


In [ ]:
# Step 12: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "Neighborhood", "2010_Total_Population"::int as pop_2010, "2020_Total_Population"::int as pop_2020, "Change_2010_to_2020_Total_Population"::int as change FROM "a8414ed5-c50f-417e-bb67-82b734660da6" ORDER BY "Change_2010_to_2020_Total_Population"::int ASC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Now I have a comprehensive picture. Let me compile everything into a thorough answer.

---

## 📊 Pittsburgh Population Trends: ~50 Years of Data

Here's what the open data reveals, assembled from multiple datasets on the **Western PA Regional Data Center (WPRDC)**:

---

### 🗓️ The Big Picture: City of Pittsburgh Population Over Decades

| Year | Population | Change from Prior Period |
|------|-----------|--------------------------|
| **1980** | ~423,938* | — |
| **2010** | ~305,704 | **-118,234 (-28%)** |
| **2020** | ~302,971 | **-2,733 (-0.9%)** |

*\*From the 1984 Community Profiles dataset (University of Pittsburgh / Health and Welfare Planning Association), which captured Pittsburgh's population around ~1980.*

> **Key finding:** Pittsburgh lost roughly **28% of its population** between ~1980 and 2010 — a dramatic decline driven by deindustrialization and the collapse of the steel industry. However, the 2010s saw near-stabilization, with the city losing only about **2,700 people (-0.9%)** over the entire decade.

---

### 📉 Long-Term Context (Beyond WPRDC Data)

For broader historical context (sourced from U.S. Census Bureau records):
- **1970**: ~520,000 (near peak)
- **1980**: ~424,000
- **1990**: ~370,000
- **2000**: ~335,000
- **2010**: ~306,000
- **2020**: ~303,000

Pittsburgh once held over **675,000 people at its 1950 peak** — and has lost more than half its population over 70 years, largely due to the steel industry collapse in the late 1970s–80s.

---

### 🏙️ Neighborhood-Level Trends (2010 → 2020)

**Top 10 Growing Neighborhoods:**

| Neighborhood | 2010 Pop | 2020 Pop | Change |
|---|---|---|---|
| Downtown (CBD) | 3,629 | 5,477 | **+1,848** |
| Shadyside | 13,915 | 15,317 | **+1,402** |
| Strip District | 616 | 1,845 | **+1,229** |
| South Side Flats | 6,597 | 7,467 | **+870** |
| Northview Heights | 1,214 | 1,769 | **+555** |
| Squirrel Hill North | 11,363 | 11,879 | **+516** |
| Allegheny Center | 933 | 1,419 | **+486** |
| South Oakland | 2,969 | 3,329 | **+360** |
| East Liberty | 5,869 | 6,187 | **+318** |
| Lower Lawrenceville | 2,341 | 2,641 | **+300** |

**Top 10 Declining Neighborhoods:**

| Neighborhood | 2010 Pop | 2020 Pop | Change |
|---|---|---|---|
| Marshall-Shadeland | 6,043 | 4,046 | **-1,997** |
| Lincoln-Lemington-Belmar | 4,883 | 3,636 | **-1,247** |
| Terrace Village | 3,228 | 2,320 | **-908** |
| Bluff | 6,600 | 5,882 | **-718** |
| Homewood North | 3,280 | 2,617 | **-663** |
| Hazelwood | 4,317 | 3,889 | **-428** |
| Perry South | 4,145 | 3,765 | **-380** |
| Beechview | 7,974 | 7,596 | **-378** |
| Upper Hill | 2,057 | 1,723 | **-334** |
| Beltzhoover | 1,925 | 1,604 | **-321** |

---

### 🔍 Key Takeaways

1. **Stabilization**: After decades of steep decline, Pittsburgh's overall population has essentially **plateaued** in the 2010s–2020s.
2. **Urban core revival**: Downtown, the Strip District, Shadyside, and Lawrenceville saw strong growth — driven by tech jobs, universities, and urban redevelopment.
3. **Persistent decline in historically Black neighborhoods**: Areas like Lincoln-Lemington-Belmar, Homewood North, and Marshall-Shadeland continue to lose population, reflecting ongoing disinvestment and economic challenges.
4. **Deindustrialization legacy**: The massive losses from 1970–2000 shaped modern Pittsburgh — smaller but increasingly knowledge-economy-driven.

---

**Sources:**
- *2020 Census Redistricting Data Extracts (PL 94-171)* — WPRDC
- *Pittsburgh SNAP Census Data (2010 neighborhood population)* — WPRDC
- *1984 Community Profiles: Allegheny County* (Univ. of Pittsburgh) — WPRDC

For deeper historical data (pre-1980), the **U.S. Census Bureau's decennial census records** are the best resource.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-16

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-16 11:13:28
- **Query**: what are the population trends in Pittsburgh over the past 50 years?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
